In [50]:
import pandas as pd
import numpy as np
import h5py
tree_file = r"C:\Users\adaly\OneDrive\Documents\DMDeficientGalaxyTNG-1\data\TNG300-1\output\tree_617680.hdf5"


with h5py.File(tree_file, "r") as f:
    snap = f["SnapNum"][:]
    subhalo_id = f["SubhaloID"][:]
    subhalo_mass = f["SubhaloMass"][:]
    subhalo_mass_in_rad = f["SubhaloMassInRad"][:]
    halfmass_rad = f["SubhaloHalfmassRad"][:]
    group_id = f["SubhaloGrNr"][:]
    mass_type = f["SubhaloMassType"][:]
    mass_in_rad_type = f["SubhaloMassInRadType"][:]
    gas_mass = mass_type[:, 0]
    dm_mass = mass_type[:, 1]
    star_mass = mass_type[:, 4]

    gas_mass_in_rad = mass_in_rad_type[:, 0]
    dm_mass_in_rad = mass_in_rad_type[:, 1]
    star_mass_in_rad = mass_in_rad_type[:, 4]
tree = pd.DataFrame({
    "SnapNum": snap,
    "SubhaloID": subhalo_id,
    "SubhaloMass": subhalo_mass,
    "SubhaloMassInRad": subhalo_mass_in_rad,
    "GasMass": gas_mass,
    "DMMass": dm_mass,
    "StarMass": star_mass,
    "GasMassInRad": gas_mass_in_rad,
    "DMMassInRad": dm_mass_in_rad,
    "StarMassInRad": star_mass_in_rad,
    "HalfmassRad": halfmass_rad,
    "GroupID": group_id
})
tree["f_DM"] = (
    tree["DMMassInRad"] /
    tree["SubhaloMassInRad"]
)
import requests
url = "https://www.tng-project.org/api/TNG300-1/snapshots/"
headers = {"api-key": "6c8ec3d465c0bcf623a06a2b4c43408c"}
response = requests.get(url, headers=headers)
response.raise_for_status()

snapshots = response.json()
snapshot_redshift = {
    snap["number"]: snap["redshift"]
    for snap in snapshots
}
tree["Redshift"] = tree["SnapNum"].map(snapshot_redshift)
print(
    tree[
        ["SnapNum", "Redshift", "SubhaloID", "f_DM"]
    ].to_string(index=False)
)

tree["Redshift"] = tree["SnapNum"].map(snapshot_redshift)

print(
    tree[
        ["SnapNum", "Redshift", "SubhaloID", "f_DM"]
    ].to_string(index=False)
)
event_data = tree.dropna(
    subset=["Redshift", "f_DM"]
).copy()

event_data = event_data.sort_values("SnapNum").reset_index(drop=True)
event_data["Delta_f_DM"] = event_data["f_DM"].diff()
biggest_drops = event_data.sort_values(
    "Delta_f_DM"
).head(10)

print(
    biggest_drops[
        [
            "SnapNum",
            "Redshift",
            "SubhaloID",
            "f_DM",
            "Delta_f_DM",
            "DMMass",
            "StarMass",
            "GasMass",
            "HalfmassRad",
            "GroupID"
        ]
    ].to_string(index=False)
)

 SnapNum     Redshift          SubhaloID     f_DM
      99 2.220446e-16 200000007700042634 0.375286
      98 9.521667e-03 200000007700042635 0.369406
      97 2.397443e-02 200000007700042636 0.408582
      96 3.372437e-02 200000007700042637 0.389125
      95 4.852363e-02 200000007700042638 0.385386
      94 5.850732e-02 200000007700042639 0.346854
      93 7.366138e-02 200000007700042640 0.406987
      92 8.388443e-02 200000007700042641 0.392110
      91 9.940180e-02 200000007700042642 0.412168
      90 1.098699e-01 200000007700042643 0.400729
      89 1.257593e-01 200000007700042644 0.410886
      88 1.418762e-01 200000007700042645 0.412977
      87 1.527488e-01 200000007700042646 0.382336
      86 1.692520e-01 200000007700042647 0.370283
      85 1.803853e-01 200000007700042648 0.356600
      84 1.972842e-01 200000007700042649 0.358354
      83 2.144250e-01 200000007700042650 0.331324
      82 2.259884e-01 200000007700042651 0.335493
      81 2.435402e-01 200000007700042652 0.331040


In [51]:
event_data["Delta_DM"] = event_data["DMMass"].diff()
event_data["Delta_star"] = event_data["StarMass"].diff()
event_data["Delta_gas"] = event_data["GasMass"].diff()

event_data["FracChange_DM"] = (
    event_data["DMMass"].diff() /
    event_data["DMMass"].shift(1)
)

event_data["FracChange_star"] = (
    event_data["StarMass"].diff() /
    event_data["StarMass"].shift(1)
)

event_data["FracChange_gas"] = (
    event_data["GasMass"].diff() /
    event_data["GasMass"].shift(1)
)

In [ ]:

biggest_drops = event_data.sort_values(
    "Delta_f_DM"
).head(10)

print(
    biggest_drops[
        [
            "SnapNum",
            "Redshift",
            "SubhaloID",
            "f_DM",
            "Delta_f_DM",
            "FracChange_DM",
            "FracChange_star",
            "FracChange_gas",
            "HalfmassRad"
        ]
    ].to_string(index=False)
)


 SnapNum  Redshift          SubhaloID     f_DM  Delta_f_DM  FracChange_DM  FracChange_star  FracChange_gas  HalfmassRad
      21  4.007945 200000007700042711 0.705211   -0.155468       0.209424        -0.045365        0.577611    33.063072
      66  0.524566 200000007700042667 0.367760   -0.134259       0.086957         0.256754        1.528639     3.373652
      33  2.002028 200000007700042700 0.630785   -0.085812      -0.190601         0.431645       -0.132353    41.084179
      38  1.604235 200000007700042695 0.702125   -0.083050      -0.966102        -0.594983       -0.966075     6.700579
      30  2.316111 200000007700042703 0.793359   -0.076598       0.527372         1.666852        0.547487    42.446957
      35  1.822689 200000007700042698 0.665342   -0.075866       0.208589         0.713916        0.038446    42.137379
      41  1.414098 200000007700042692 0.731113   -0.061913      -0.265035         0.068612       -0.296613    28.697849
      46  1.154603 200000007700042687 0.

In [53]:
event_snap = 66

window = event_data[
    (event_data["SnapNum"] >= event_snap - 3) &
    (event_data["SnapNum"] <= event_snap + 3)
].copy()

print(
    window[
        [
            "SnapNum",
            "Redshift",
            "SubhaloID",
            "f_DM",
            "DMMass",
            "StarMass",
            "GasMass",
            "HalfmassRad",
            "GroupID"
        ]
    ].to_string(index=False)
)

 SnapNum  Redshift          SubhaloID     f_DM   DMMass  StarMass  GasMass  HalfmassRad  GroupID
      63  0.598543 200000007700042670 0.546013 0.920172  0.370020 0.443938     5.612443     6166
      64  0.575981 200000007700042669 0.519817 0.736934  0.351516 0.249746     4.006640     6167
      65  0.546392 200000007700042668 0.502019 0.549713  0.300795 0.219637     3.113211      823
      66  0.524566 200000007700042667 0.367760 0.597514  0.378025 0.555384     3.373652      749
      67  0.503048 200000007700042666 0.383106 0.545730  0.384533 0.309879     2.751270      763
      68  0.481833 200000007700042665 0.417306 0.537763  0.384615 0.184560     2.863821      786
      69  0.460918 200000007700042664 0.409226 0.466061  0.362578 0.157085     2.607845      768


In [54]:
window = event_data[
    (event_data["SnapNum"] >= 63) &
    (event_data["SnapNum"] <= 69)
].copy()

print(
    window[
        [
            "SnapNum",
            "Redshift",
            "f_DM",
            "DMMassInRad",
            "StarMassInRad",
            "GasMassInRad",
            "HalfmassRad"
        ]
    ].to_string(index=False)
)

 SnapNum  Redshift     f_DM  DMMassInRad  StarMassInRad  GasMassInRad  HalfmassRad
      63  0.598543 0.546013     0.577597       0.257625      0.222623     5.612443
      64  0.575981 0.519817     0.442160       0.253248      0.155200     4.006640
      65  0.546392 0.502019     0.382409       0.246121      0.133212     3.113211
      66  0.524566 0.367760     0.366475       0.291514      0.338519     3.373652
      67  0.503048 0.383106     0.314691       0.298594      0.208134     2.751270
      68  0.481833 0.417306     0.298757       0.301607      0.115555     2.863821
      69  0.460918 0.409226     0.270873       0.291088      0.099955     2.607845


In [55]:
window["Delta_DM_inRad"] = window["DMMassInRad"].diff()
window["Delta_star_inRad"] = window["StarMassInRad"].diff()
window["Delta_gas_inRad"] = window["GasMassInRad"].diff()

print(
    window[
        [
            "SnapNum",
            "Redshift",
            "f_DM",
            "DMMassInRad",
            "StarMassInRad",
            "GasMassInRad",
            "Delta_DM_inRad",
            "Delta_star_inRad",
            "Delta_gas_inRad"
        ]
    ].to_string(index=False)
)


 SnapNum  Redshift     f_DM  DMMassInRad  StarMassInRad  GasMassInRad  Delta_DM_inRad  Delta_star_inRad  Delta_gas_inRad
      63  0.598543 0.546013     0.577597       0.257625      0.222623             NaN               NaN              NaN
      64  0.575981 0.519817     0.442160       0.253248      0.155200       -0.135437         -0.004378        -0.067422
      65  0.546392 0.502019     0.382409       0.246121      0.133212       -0.059751         -0.007127        -0.021988
      66  0.524566 0.367760     0.366475       0.291514      0.338519       -0.015934          0.045393         0.205307
      67  0.503048 0.383106     0.314691       0.298594      0.208134       -0.051785          0.007080        -0.130385
      68  0.481833 0.417306     0.298757       0.301607      0.115555       -0.015934          0.003014        -0.092579
      69  0.460918 0.409226     0.270873       0.291088      0.099955       -0.027884         -0.010520        -0.015599


In [56]:
event_snap = 66

window = event_data[
    (event_data["SnapNum"] >= event_snap - 3) &
    (event_data["SnapNum"] <= event_snap + 3)
].copy()

print(
    window[
        [
            "SnapNum",
            "SubhaloID",
        ]
    ].to_string(index=False)
)

 SnapNum          SubhaloID
      63 200000007700042670
      64 200000007700042669
      65 200000007700042668
      66 200000007700042667
      67 200000007700042666
      68 200000007700042665
      69 200000007700042664


In [57]:
import h5py
import pandas as pd

tree_file = r"C:\Users\adaly\OneDrive\Documents\DMDeficientGalaxyTNG-1\data\TNG300-1\output\tree_738558.hdf5"

with h5py.File(tree_file, "r") as f:
    snap = f["SnapNum"][:]
    subhalo_id = f["SubhaloID"][:]
    subfind_id = f["SubfindID"][:]
    f_dm = f["SubhaloMassInRadType"][:, 1] / f["SubhaloMassInRad"][:]
    halfmass_rad = f["SubhaloHalfmassRad"][:]

event_data = pd.DataFrame({
    "SnapNum": snap,
    "SubhaloID": subhalo_id,
    "SubfindID": subfind_id,
    "f_DM": f_dm,
    "HalfmassRad": halfmass_rad
})
event_snaps = event_data[
    event_data["SnapNum"].isin([63, 64, 65, 66])
].copy()

print(event_snaps.to_string(index=False))
for _, row in event_snaps.iterrows():

    snap = int(row["SnapNum"])
    subfind_id = int(row["SubhaloID"])

    url = (
        f"https://www.tng-project.org/api/TNG300-1/"
        f"snapshots/{snap}/subhalos/{subfind_id}/"
    )

    response = requests.get(url, headers=headers)

    print(snap, subfind_id, response.status_code)
    print(response.json())

 SnapNum         SubhaloID  SubfindID     f_DM  HalfmassRad
      66 50000000100521884     382115 0.512941    16.989349
      65 50000000100521885     374065 0.523803    19.669220
      64 50000000100521886     364735 0.535245    17.921209
      63 50000000100521887     357250 0.542193    18.830345


C:\Users\adaly\AppData\Local\Temp\ipykernel_24284\3728786185.py:10: RuntimeWarning: invalid value encountered in divide
  f_dm = f["SubhaloMassInRadType"][:, 1] / f["SubhaloMassInRad"][:]


66 50000000100521888 404
{'detail': 'Not found.'}
65 50000000100521888 404
{'detail': 'Not found.'}
64 50000000100521888 404
{'detail': 'Not found.'}
63 50000000100521888 404
{'detail': 'Not found.'}


In [58]:
import requests

API_KEY = "6c8ec3d465c0bcf623a06a2b4c43408c "

headers = {
    "api-key": API_KEY
}

for _, row in event_snaps.iterrows():

    snap = int(row["SnapNum"])
    subfind_id = int(row["SubfindID"])

    url = (
        f"https://www.tng-project.org/api/TNG300-1/"
        f"snapshots/{snap}/subhalos/{subfind_id}/"
    )

    response = requests.get(url, headers=headers)

    print(
        f"Snap {snap}: "
        f"SubfindID {subfind_id}: "
        f"HTTP {response.status_code}"
    )

    if response.status_code == 200:
        data = response.json()
        print(data)
    else:
        print(response.text)

Snap 66: SubfindID 382115: HTTP 200
{'snap': 66, 'id': 382115, 'bhmdot': 0.00171116, 'cm_x': 44581.1, 'cm_y': 162441.0, 'cm_z': 47624.0, 'gasmetallicity': 0.0221359, 'gasmetallicityhalfrad': 0.0272297, 'gasmetallicitymaxrad': 0.0235245, 'gasmetallicitysfr': 0.0197352, 'gasmetallicitysfrweighted': 0.0206857, 'pos_x': 44581.3, 'pos_y': 162441.0, 'pos_z': 47623.7, 'halfmassrad': 16.9893, 'halfmassrad_gas': 9.2961, 'halfmassrad_dm': 21.0769, 'halfmassrad_stars': 4.3834, 'halfmassrad_bhs': 0.0, 'len': 13553, 'len_gas': 2264, 'len_dm': 5037, 'len_stars': 6251, 'len_bhs': 1, 'mass': 25.5862, 'mass_gas': 1.97689, 'mass_dm': 20.0645, 'mass_stars': 3.53172, 'mass_bhs': 0.013042, 'massinhalfrad': 3.53007, 'massinhalfrad_gas': 0.261463, 'massinhalfrad_dm': 1.4898, 'massinhalfrad_stars': 1.76577, 'massinhalfrad_bhs': 0.013042, 'massinmaxrad': 5.42144, 'massinmaxrad_gas': 0.669864, 'massinmaxrad_dm': 2.48566, 'massinmaxrad_stars': 2.25288, 'massinmaxrad_bhs': 0.013042, 'massinrad': 7.23777, 'massinr

In [59]:
import requests

API_KEY = "6c8ec3d465c0bcf623a06a2b4c43408c"

headers = {
    "api-key": API_KEY
}

url = (
    "https://www.tng-project.org/api/TNG300-1/"
    "snapshots/33/subhalos/65188/sublink/simple.json"
)

response = requests.get(url, headers=headers)

print(response.status_code)

data = response.json()

print(data)
print("Mergers:")
for merger in data.get("Mergers", []):
    print(merger)

200
{'Main': [[49, 226199], [48, 212670], [47, 200545], [46, 185183], [45, 168962], [44, 171306], [43, 156178], [42, 139993], [41, 136598], [40, 121239], [39, 114370], [38, 104801], [37, 101541], [36, 93555], [35, 85946], [34, 75291], [33, 65137], [32, 61899], [31, 52068], [30, 43646], [29, 57099], [28, 53806], [27, 65070], [26, 55621], [25, 48662], [24, 42483], [23, 34127], [22, 28635], [21, 70765], [20, 180948], [19, 146089], [18, 184582], [17, 166749], [16, 261826], [15, 251129], [14, 224856], [13, 213044], [12, 213669], [11, 187912], [10, 161686], [9, 135900], [8, 137449], [7, 165050], [6, 109145], [5, 98827], [4, 90718], [3, 100743], [2, 63023]], 'Mergers': [[50, 38737], [48, 212676], [48, 212812], [48, 212856], [48, 212841], [48, 212855], [48, 14856223], [48, 212859], [48, 212860], [48, 212842], [48, 212858], [47, 17352875], [48, 212866], [48, 212861], [47, 200704], [47, 6200152], [46, 9616507], [46, 14188331], [46, 15261542], [47, 200695], [47, 200733], [46, 185376], [46, 185384

In [60]:
snapshots = {
    63: 357250,
    64: 364735,
    65: 374065,
    66: 382115
}

for snap, halo_id in snapshots.items():

    url = (
        f"https://www.tng-project.org/api/TNG300-1/"
        f"snapshots/{snap}/halos/{halo_id}/"
    )

    response = requests.get(url, headers=headers)

    print(f"\nSnap {snap}, halo {halo_id}: {response.status_code}")

    if response.status_code == 200:
        print(response.json())
    else:
        print(response.text)


Snap 63, halo 357250: 200
{'snap': 63, 'halo_id': 357250, 'meta': {'simulation': 'http://www.tng-project.org/api/TNG300-1/', 'snapshot': 'http://www.tng-project.org/api/TNG300-1/snapshots/63/', 'url': 'http://www.tng-project.org/api/TNG300-1/snapshots/63/halos/357250/', 'info': 'http://www.tng-project.org/api/TNG300-1/snapshots/63/halos/357250/info.json'}, 'vis': {'halo_gas_dens': 'http://www.tng-project.org/api/TNG300-1/snapshots/63/halos/357250/vis.png?partType=gas', 'halo_gas_temp': 'http://www.tng-project.org/api/TNG300-1/snapshots/63/halos/357250/vis.png?partType=gas&partField=temp', 'halo_dm_dens': 'http://www.tng-project.org/api/TNG300-1/snapshots/63/halos/357250/vis.png?partType=dm', 'halo_stellar_dens': 'http://www.tng-project.org/api/TNG300-1/snapshots/63/halos/357250/vis.png?partType=stars', 'galaxy_gas_dens': 'http://www.tng-project.org/api/TNG300-1/snapshots/63/halos/357250/vis.png?partType=gas&size=5.0&sizeType=rHalfMassStars', 'galaxy_gas_dens_faceon': 'http://www.tng-p

In [61]:
hosts = {
    63: 357250,
    64: 364735,
    65: 374065,
}

for snap, halo_id in hosts.items():

    url = (
        f"https://www.tng-project.org/api/TNG300-1/"
        f"snapshots/{snap}/halos/{halo_id}/info.json"
    )

    r = requests.get(url, headers=headers)

    print(f"\n--- Snap {snap}, Group {halo_id} ---")
    print("Status:", r.status_code)

    if r.status_code == 200:
        d = r.json()

        print("Group position:", d["GroupPos"])
        print("R200:", d["Group_R_Crit200"])
        print("M200:", d["Group_M_Crit200"])


--- Snap 63, Group 357250 ---
Status: 200
Group position: [81493.2734375, 59618.09375, 52575.24609375]
R200: 72.3638687133789
M200: 4.21104621887207

--- Snap 64, Group 364735 ---
Status: 200
Group position: [7245.474609375, 105076.4765625, 132574.890625]
R200: 46.71405792236328
M200: 1.1503134965896606

--- Snap 65, Group 374065 ---
Status: 200
Group position: [60831.63671875, 98037.3203125, 17843.990234375]
R200: 79.45356750488281
M200: 5.780410289764404


In [64]:
subs = {
    63: 357250,
    64: 364735,
    65: 374065,
}

for snap, subfind in subs.items():

    url = (
        f"https://www.tng-project.org/api/TNG300-1/"
        f"snapshots/{snap}/subhalos/{subfind}/info.json"
    )

    r = requests.get(url, headers=headers)

    print(f"\n--- Snap {snap}, SubfindID {subfind} ---")
    print("Status:", r.status_code)

    if r.status_code == 200:
        d = r.json()

        print("Position:", d["SubhaloPos"])
        print("Group:", d["SubhaloGrNr"])
        print("Parent:", d.get("SubhaloParent"))
        print("Primary:", d.get("SubhaloPrimaryFlag"))
        print("Mass:", d["SubhaloMass"])
        print("HalfmassRad:", d["SubhaloHalfmassRad"])


--- Snap 63, SubfindID 357250 ---
Status: 200
Position: [44523.1953125, 162349.765625, 47581.2734375]
Group: 719
Parent: 0
Primary: None
Mass: 28.75989532470703
HalfmassRad: 18.830345153808594

--- Snap 64, SubfindID 364735 ---
Status: 200
Position: [44504.03125, 162377.234375, 47596.29296875]
Group: 717
Parent: 0
Primary: None
Mass: 27.185497283935547
HalfmassRad: 17.92120933532715

--- Snap 65, SubfindID 374065 ---
Status: 200
Position: [44527.98828125, 162409.921875, 47612.046875]
Group: 718
Parent: 0
Primary: None
Mass: 28.983694076538086
HalfmassRad: 19.669219970703125


In [ ]:
import requests

API_KEY = "6c8ec3d465c0bcf623a06a2b4c43408c "
headers = {"api-key": API_KEY}

mergers = {
    30: [893799],
    32: [697207, 697203]
}

for snap, subhalos in mergers.items():

    print(f"\n========== SNAPSHOT {snap} ==========")

    for subfind in subhalos:

        url = (
            f"https://www.tng-project.org/api/TNG300-1/"
            f"snapshots/{snap}/subhalos/{subfind}/info.json"
        )

        r = requests.get(url, headers=headers)

        print(f"\n--- Merger SubfindID {subfind} ---")
        print("Status:", r.status_code)

        if r.status_code == 200:

            d = r.json()

            print("Position:", d.get("SubhaloPos"))
            print("Group:", d.get("SubhaloGrNr"))
            print("Mass:", d.get("SubhaloMass"))
            print("MassType:", d.get("SubhaloMassType"))
            print("HalfmassRad:", d.get("SubhaloHalfmassRad"))
            print("Stellar mass:", d.get("SubhaloMassType", [None]*5)[4])
            print("DM mass:", d.get("SubhaloMassType", [None]*2)[1])

        else:
            print(r.text)


========== SNAPSHOT 30 ==========

--- Merger SubfindID 893799 ---
Status: 200
Position: [44194.15234375, 22613.734375, 50884.015625]
Group: 77922
Mass: 0.09161883592605591
MassType: [0.0, 0.09161883592605591, 0.0, 0.0, 0.0, 0.0]
HalfmassRad: 12.239214897155762
Stellar mass: 0.0
DM mass: 0.09161883592605591

========== SNAPSHOT 32 ==========

--- Merger SubfindID 697207 ---
Status: 200
Position: [44106.8671875, 22967.73828125, 50937.83203125]
Group: 35177
Mass: 0.09284117072820663
MassType: [0.0012223361991345882, 0.09161883592605591, 0.0, 0.0, 0.0, 0.0]
HalfmassRad: 12.833819389343262
Stellar mass: 0.0
DM mass: 0.09161883592605591

--- Merger SubfindID 697203 ---
Status: 200
Position: [43786.58203125, 22791.9765625, 50976.2890625]
Group: 35177
Mass: 0.13941995799541473
MassType: [0.0, 0.13941995799541473, 0.0, 0.0, 0.0, 0.0]
HalfmassRad: 15.173460006713867
Stellar mass: 0.0
DM mass: 0.13941995799541473
